# Linear Regression with NeuraCommand

This notebook demonstrates using NeuraCommand for simple linear regression.

We'll fit the function: **y = 2x + 1**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '../src')

from neura_command import Sequential, Dense, Adam, MSELoss

# For reproducibility
np.random.seed(42)

## 1. Generate Data

Create noisy samples from `y = 2x + 1`

In [ ]:
# Generate training data
n_samples = 200
X = np.random.uniform(-1, 1, (n_samples, 1)).astype(np.float32)
y_true = 2 * X + 1
y = y_true + np.random.normal(0, 0.1, (n_samples, 1)).astype(np.float32)  # Add noise

# Split data
split = int(0.8 * n_samples)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Visualize
plt.figure(figsize=(10, 5))
plt.scatter(X_train, y_train, alpha=0.6, label='Training data')
plt.scatter(X_test, y_test, alpha=0.6, label='Test data')
plt.plot(np.sort(X.flatten()), 2 * np.sort(X.flatten()) + 1, 'r--', linewidth=2, label='True: y=2x+1')
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.title('Linear Regression Data')
plt.grid(True, alpha=0.3)
plt.show()

## 2. Create and Train Model

For linear regression, we use a single Dense layer with no activation (linear output).

In [ ]:
# Create a simple linear model (no hidden layers, no activation)
model = Sequential(
    layers=[Dense(1, 1)],  # 1 input -> 1 output
    optimizer=Adam(learning_rate=0.1),
    loss=MSELoss(),
)

print(model.summary())

In [ ]:
# Train the model
history = model.train(
    X_train, y_train,
    epochs=100,
    batch_size=16,
    validation_data=(X_test, y_test),
    verbose=False,
)

# Get final loss
final_train_loss = history['loss'][-1]
final_val_loss = history['val_loss'][-1]
print(f"Final training loss: {final_train_loss:.6f}")
print(f"Final validation loss: {final_val_loss:.6f}")

## 3. Evaluate Results

In [ ]:
# Get learned parameters
dense_layer = model.layers[0]
learned_weight = dense_layer.weights[0, 0]
learned_bias = dense_layer.bias[0, 0]

print(f"True function: y = 2.00x + 1.00")
print(f"Learned:       y = {learned_weight:.2f}x + {learned_bias:.2f}")

In [ ]:
# Make predictions
predictions = model.predict(X_test)

# Calculate metrics
mse = np.mean((predictions - y_test) ** 2)
mae = np.mean(np.abs(predictions - y_test))
r2 = 1 - (np.sum((y_test - predictions) ** 2) / np.sum((y_test - np.mean(y_test)) ** 2))

print(f"Mean Squared Error:  {mse:.6f}")
print(f"Mean Absolute Error: {mae:.6f}")
print(f"R² Score:            {r2:.4f}")

In [ ]:
# Visualize results
plt.figure(figsize=(12, 5))

# Plot 1: Data and fit
plt.subplot(1, 2, 1)
plt.scatter(X_train, y_train, alpha=0.4, label='Training data')
plt.scatter(X_test, y_test, alpha=0.4, label='Test data')

x_line = np.linspace(-1, 1, 100).reshape(-1, 1)
plt.plot(x_line, 2 * x_line + 1, 'g--', linewidth=2, label='True: y=2x+1')
plt.plot(x_line, model.predict(x_line.astype(np.float32)), 'r-', linewidth=2, 
         label=f'Learned: y={learned_weight:.2f}x+{learned_bias:.2f}')

plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.title('Linear Regression Fit')
plt.grid(True, alpha=0.3)

# Plot 2: Loss history
plt.subplot(1, 2, 2)
plt.plot(history['loss'], label='Training Loss')
plt.plot(history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.title('Training History')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Multi-feature Linear Regression

Let's try a more complex example: **y = 3x₁ - 2x₂ + 0.5x₃ + 1**

In [ ]:
# Generate multi-feature data
n_samples = 500
X_multi = np.random.uniform(-1, 1, (n_samples, 3)).astype(np.float32)
y_multi = (3 * X_multi[:, 0:1] - 2 * X_multi[:, 1:2] + 0.5 * X_multi[:, 2:3] + 1)
y_multi += np.random.normal(0, 0.1, y_multi.shape).astype(np.float32)  # Add noise

# Split
split = int(0.8 * n_samples)
X_train_m, X_test_m = X_multi[:split], X_multi[split:]
y_train_m, y_test_m = y_multi[:split], y_multi[split:]

print(f"Input shape: {X_train_m.shape}")
print(f"Output shape: {y_train_m.shape}")

In [ ]:
# Create and train model
model_multi = Sequential(
    layers=[Dense(3, 1)],  # 3 inputs -> 1 output
    optimizer=Adam(learning_rate=0.1),
    loss=MSELoss(),
)

model_multi.train(
    X_train_m, y_train_m,
    epochs=100,
    batch_size=32,
    verbose=False,
)

# Get learned parameters
weights = model_multi.layers[0].weights.flatten()
bias = model_multi.layers[0].bias.flatten()[0]

print(f"True:    y = 3.00x₁ - 2.00x₂ + 0.50x₃ + 1.00")
print(f"Learned: y = {weights[0]:.2f}x₁ + {weights[1]:.2f}x₂ + {weights[2]:.2f}x₃ + {bias:.2f}")

In [ ]:
# Evaluate
predictions_m = model_multi.predict(X_test_m)
mse_m = np.mean((predictions_m - y_test_m) ** 2)
r2_m = 1 - (np.sum((y_test_m - predictions_m) ** 2) / np.sum((y_test_m - np.mean(y_test_m)) ** 2))

print(f"MSE:      {mse_m:.6f}")
print(f"R² Score: {r2_m:.4f}")

# Scatter plot of predictions vs actual
plt.figure(figsize=(6, 6))
plt.scatter(y_test_m, predictions_m, alpha=0.5)
plt.plot([-3, 5], [-3, 5], 'r--', linewidth=2, label='Perfect prediction')
plt.xlabel('Actual y')
plt.ylabel('Predicted y')
plt.title(f'Multi-feature Linear Regression (R²={r2_m:.3f})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.show()

## Summary

- For **linear regression**, use a single `Dense` layer with no activation
- NeuraCommand successfully learns linear relationships
- Works with single or multiple input features
- Use `MSELoss()` for regression problems